In [1]:
cd /content/drive/MyDrive

/content/drive/MyDrive


In [2]:
# 1. Check GPU Availability and CUDA Version
# This step verifies that a GPU is available and shows its details,
# including the CUDA version. This is important for selecting the correct
# PaddlePaddle GPU package.

print("--- Checking GPU and CUDA Version ---")
!nvidia-smi
!nvcc --version

# Based on the output of `nvcc --version`, you might need to adjust the CUDA version
# in the PaddlePaddle installation command below (e.g., cu118, cu126, cu129).
# Colab typically has a recent CUDA version (e.g., 11.8 or 12.x).

# 2. Install PaddlePaddle with GPU Support (Version 3.1.0)
# This installs the PaddlePaddle deep learning framework, which is a prerequisite for PaddleOCR.
# We are installing the GPU-enabled version 3.1.0, compatible with CUDA 12.x.
# If your Colab runtime provides a different CUDA version (e.g., 11.x),
# replace 'cu126' with 'cu118' or 'cu129' accordingly based on PaddlePaddle's official documentation.

print("\n--- Installing PaddlePaddle GPU 3.1.0 ---")
!python -m pip install paddlepaddle-gpu==3.1.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/

# 3. Install PaddleOCR (Version 3.2.0)
# This installs the PaddleOCR library itself.

print("\n--- Installing PaddleOCR 3.2.0 ---")
!pip install paddleocr==3.2.0

# 4. Install Additional Dependencies
# These are common dependencies required for PaddleOCR's functionality and image processing.

print("\n--- Installing additional dependencies ---")
!pip install opencv-python imgaug pyclipper shapely pillow

# 5. Download Chinese Font for Visualization
# PaddleOCR's `draw_ocr` function requires a font that supports Chinese characters
# for proper visualization of results.

print("\n--- Downloading Chinese font ---")
!wget -P /content/ https://raw.githubusercontent.com/PaddlePaddle/PaddleOCR/release/2.6/doc/fonts/simfang.ttf

# 6. Example Usage: Perform OCR on a Chinese Image
# This section demonstrates how to use PaddleOCR to detect and recognize
# Chinese text from an image.

print("\n--- Running PaddleOCR Example ---")

--- Checking GPU and CUDA Version ---
Fri Mar 27 07:49:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+---------

In [ ]:
import os
import csv
from paddleocr import PaddleOCR
from PIL import Image
import numpy as np # Used for convenient bounding box calculations
import requests
import re

# --- Configuration ---
input_image_folder = '/content/drive/MyDrive/连环画/Ref002/JPEG/Core'
output_csv_filename = '/content/drive/MyDrive/连环画/Ref002/CSV/OCR_Raw_Test'

# Define supported image extensions
IMAGE_EXTENSIONS = ('.png', '.jpg', '.jpeg', '.bmp', '.tiff', '.webp')

# Threshold for grouping lines vertically (in pixels)
# If the difference in top Y-coordinates of consecutive lines is within this threshold, they are grouped.
GROUPING_THRESHOLD_Y = 20

# Ratio to define the *global* footer region for identification (e.g., 0.975 means bottom 2.5% of the entire image).
# Text lines with a min_y greater than or equal to (image_height * FOOTER_Y_THRESHOLD_RATIO) will be identified as global footnotes.
FOOTER_Y_THRESHOLD_RATIO = 0.975

# Ratio to define an *inner* footer region within each ROI for identification (e.g., 0.15 means bottom 15% of the ROI's height).
# Text lines with a min_y greater than or equal to (ROI_end_y - (ROI_height * ROI_INNER_FOOTER_RATIO)) will be identified as ROI footnotes.
ROI_INNER_FOOTER_RATIO = 0.15

KEYWORDS_TO_REMOVE = [
]

def remove_keywords(text, keywords):
    cleaned_text = text
    for keyword in keywords:
        # Create a regex pattern to match the whole word (case-insensitive)
        # \b ensures whole word match, re.IGNORECASE for case-insensitivity
        pattern = r'\b' + re.escape(keyword) + r'\b'
        cleaned_text = re.sub(pattern, '', cleaned_text, flags=re.IGNORECASE).strip()
        # Remove any extra spaces left by removal (e.g., "word1  word2")
        cleaned_text = re.sub(r'\s+', ' ', cleaned_text).strip()
    return cleaned_text

# Helper function to get min/max coordinates from a bounding box
def get_bbox_coords(bbox_points):
    points = np.array(bbox_points)
    min_x, min_y = np.min(points, axis=0)
    max_x, max_y = np.max(points, axis=0)
    return min_x, min_y, max_x, max_y

# Helper function to check if a bounding box's center is within an ROI
def is_bbox_center_in_roi(bbox, roi):
    _, min_y, _, max_y = get_bbox_coords(bbox)
    bbox_center_y = (min_y + max_y) / 2
    # ROI is [roi_x_start, roi_y_start, roi_x_end, roi_y_end]
    return roi[1] <= bbox_center_y <= roi[3]

# --- Step 2: Initialize PaddleOCR ---
# 'lang="ch"' loads the Chinese model which handles both simplified and traditional.
# 'use_textline_orientation=True' improves accuracy by correcting text orientation.
print("\n--- Initializing PaddleOCR (this may download models if first run) ---")
ocr = PaddleOCR(use_textline_orientation=True, lang="ch")

# --- Step 3: Prepare CSV file for writing ---
# The output CSV file path should be joined to /content/drive/MyDrive, not just /content/
csv_file_path = os.path.join('/content/drive/MyDrive/连环画/Ref002/CSV', output_csv_filename.split('/')[-1])
csv_data = []
# Updated CSV header to include 'Column' and 'Footnote_Text' for multi-column layout and footnotes
csv_header = ['Page_Number', 'ROI', 'Column', 'Text_Line', 'Footnote_Text']
csv_data.append(csv_header)

# --- Step 4: Iterate through images, perform OCR, and collect data ---
print(f"\n--- Starting OCR for images in: {input_image_folder} ---")
processed_images_count = 0


# Get list of all files in the directory
all_files_in_folder = os.listdir(input_image_folder)

# Filter and SORT the image files
image_files = sorted([f for f in all_files_in_folder if f.lower().endswith(IMAGE_EXTENSIONS)])

for filename in image_files: # Use the sorted list here
    image_path = os.path.join(input_image_folder, filename)
    print(f"Processing: {filename}")
    try:
        # Load image to get dimensions for ROI calculation
        with Image.open(image_path) as img:
            image_width, image_height = img.size

        # Define ROIs dynamically
        # ROI 1: Bottom 42% of the upper half
        # Upper half ends at image_height / 2
        # Bottom 42% means from image_height/2 * (1 - 0.42) to image_height/2
        roi1_y_start = image_height / 2 * (1 - 0.42)
        roi1_y_end = image_height / 2
        roi1 = [0, roi1_y_start, image_width, roi1_y_end] # [x_start, y_start, x_end, y_end]

        # ROI 2: Bottom 42% of the lower half
        # Lower half starts at image_height / 2 and ends at image_height
        # Bottom 42% means from image_height - (image_height/2 * 0.42) to image_height
        roi2_y_start = image_height - (image_height / 2 * 0.42)
        roi2_y_end = image_height
        roi2 = [0, roi2_y_start, image_width, roi2_y_end]

        ocr_output = ocr.predict(image_path)

        all_lines_raw = []
        image_footnote_lines = [] # List to store footnotes for the current image

        if ocr_output and isinstance(ocr_output, list) and len(ocr_output) > 0:
            for page_result_dict in ocr_output:
                if isinstance(page_result_dict, dict) and \
                  'rec_polys' in page_result_dict and \
                  'rec_texts' in page_result_dict and \
                  'rec_scores' in page_result_dict:

                    rec_polys = page_result_dict['rec_polys']
                    rec_texts = page_result_dict['rec_texts']
                    rec_scores = page_result_dict['rec_scores']

                    for i in range(len(rec_polys)):
                        bounding_box = rec_polys[i]
                        text = rec_texts[i]
                        confidence = rec_scores[i]

                        # Calculate min_y for the bounding box
                        min_x, min_y, max_x, max_y = get_bbox_coords(bounding_box)

                        if not isinstance(text, str):
                            text = str(text)

                        # Identify as global footer text (footnote)
                        if min_y >= image_height * FOOTER_Y_THRESHOLD_RATIO:
                            print(f"  Identified GLOBAL footer text (footnote): '{text}' (min_y={min_y:.0f}, threshold_y={(image_height * FOOTER_Y_THRESHOLD_RATIO):.0f})")
                            image_footnote_lines.append({
                                'bbox': bounding_box,
                                'text': text,
                                'confidence': confidence
                            })
                            continue # Skip to the next line as it's a footnote

                        all_lines_raw.append({
                            'bbox': bounding_box,
                            'text': text,
                            'confidence': confidence
                        })
                else:
                    print(f"  Warning: Unexpected dictionary structure in OCR output for {filename}. Skipping block.")
                    print(f"  Block content: {page_result_dict}")
        else:
            print(f"  Warning: OCR output is empty or not in expected list format for {filename}. No text extracted.")

        # Calculate column width for subdividing ROIs
        column_width = image_width / 4

        # Initialize lists for lines in each column within each ROI
        roi1_cols_lines = [[] for _ in range(4)] # For ROI1: Col1, Col2, Col3, Col4
        roi2_cols_lines = [[] for _ in range(4)] # For ROI2: Col1, Col2, Col3, Col4

        # Distribute lines into ROIs and then into columns
        for line_data in all_lines_raw:
            min_x, min_y, max_x, max_y = get_bbox_coords(line_data['bbox']) # Get all coords for min_y, max_y
            bbox_center_x = (min_x + max_x) / 2 # Use center X-coordinate for column assignment

            # Determine raw column index (0-3)
            col_idx = int(bbox_center_x / column_width)

            # NEW FILTERING: Skip line if its horizontal center is outside the 4 columns
            if not (0 <= col_idx < 4):
                print(f"  Skipping text horizontally outside columns: '{line_data['text']}' (bbox_center_x={bbox_center_x:.0f}, col_idx={col_idx})")
                continue # Skip this line as it's horizontally outside the expected columns

            # Check for ROI1
            if is_bbox_center_in_roi(line_data['bbox'], roi1):
                roi_height = roi1[3] - roi1[1] # height of ROI1
                roi_footer_start_y = roi1[3] - (roi_height * ROI_INNER_FOOTER_RATIO)
                if min_y >= roi_footer_start_y:
                    print(f"  Identified ROI1 inner footer text (footnote): '{line_data['text']}' (min_y={min_y:.0f}, ROI1_footer_threshold_y={roi_footer_start_y:.0f})")
                    image_footnote_lines.append(line_data) # Add to footnotes
                    continue # Skip to the next line as it's a footnote

                roi1_cols_lines[col_idx].append(line_data)
            # Check for ROI2
            elif is_bbox_center_in_roi(line_data['bbox'], roi2):
                roi_height = roi2[3] - roi2[1] # height of ROI2
                roi_footer_start_y = roi2[3] - (roi_height * ROI_INNER_FOOTER_RATIO)
                if min_y >= roi_footer_start_y:
                    print(f"  Identified ROI2 inner footer text (footnote): '{line_data['text']}' (min_y={min_y:.0f}, ROI2_footer_threshold_y={roi_footer_start_y:.0f})")
                    image_footnote_lines.append(line_data) # Add to footnotes
                    continue # Skip to the next line as it's a footnote

                roi2_cols_lines[col_idx].append(line_data)

        # --- Group lines within each column independently ---
        def group_lines_vertically(lines, threshold_y):
            if not lines:
                return []

            # Sort lines by their top-most y-coordinate first
            lines.sort(key=lambda x: np.min(np.array(x['bbox'])[:, 1]))

            grouped_lines = []
            if not lines: # Handle empty lines list after sorting
                return []

            current_group = [lines[0]]

            for i in range(1, len(lines)):
                prev_line_bbox = np.array(current_group[-1]['bbox'])
                current_line_bbox = np.array(lines[i]['bbox'])

                prev_min_y = np.min(prev_line_bbox[:, 1])
                current_min_y = np.min(current_line_bbox[:, 1])

                if abs(current_min_y - prev_min_y) <= threshold_y:
                    current_group.append(lines[i])
                else:
                    grouped_lines.append(current_group)
                    current_group = [lines[i]]

            if current_group: # Add the last group
                grouped_lines.append(current_group)

            merged_output = []
            for group in grouped_lines:
                # Sort lines within the current group by their X-coordinate
                group.sort(key=lambda x: np.min(np.array(x['bbox'])[:, 0]))

                merged_text = " ".join([line['text'] for line in group]) # Join with space
                merged_confidence = np.mean([line['confidence'] for line in group]) # Average confidence

                # Calculate the merged bounding box
                all_x_coords = []
                all_y_coords = []
                for line in group:
                    bbox_points = np.array(line['bbox'])
                    all_x_coords.extend(bbox_points[:, 0])
                    all_y_coords.extend(bbox_points[:, 1])

                min_x, min_y = np.min(all_x_coords), np.min(all_y_coords)
                max_x, max_y = np.max(all_x_coords), np.max(all_y_coords)

                merged_output.append({
                    'bbox': [[min_x, min_y], [max_x, min_y], [max_x, max_y], [min_x, max_y]],
                    'text': merged_text,
                    'confidence': merged_confidence
                })
            return merged_output

        # List to store all final processed lines for the current image
        final_processed_lines_for_csv = []

        # Process ROI1 columns
        for i, col_lines in enumerate(roi1_cols_lines):
            grouped_col_lines = group_lines_vertically(col_lines, GROUPING_THRESHOLD_Y)
            for line_info in grouped_col_lines:
                line_info['roi_label'] = 'ROI1'
                line_info['column_label'] = f'Col{i+1}'
                final_processed_lines_for_csv.append(line_info)

        # Process ROI2 columns
        for i, col_lines in enumerate(roi2_cols_lines):
            grouped_col_lines = group_lines_vertically(col_lines, GROUPING_THRESHOLD_Y)
            for line_info in grouped_col_lines:
                line_info['roi_label'] = 'ROI2'
                line_info['column_label'] = f'Col{i+1}'
                final_processed_lines_for_csv.append(line_info)

        # Prepare abbreviated filename
        last_dot_index = filename.rfind('.')
        filename_without_extension = filename[:last_dot_index] if last_dot_index != -1 else filename
        abbreviated_filename = filename_without_extension[-3:]

        # Add results to csv_data from the fully processed lines (main content)
        for line_data in final_processed_lines_for_csv:
            cleaned_text = remove_keywords(line_data['text'], KEYWORDS_TO_REMOVE)
            if cleaned_text: # Only add the line if the text is not empty after cleaning
                csv_data.append([
                    abbreviated_filename,
                    line_data['roi_label'],
                    line_data['column_label'],
                    cleaned_text,
                    '' # Empty for Footnote_Text
                ])

        # Now add the identified footnotes for this image
        # Sort footnotes by their Y-coordinate to maintain order
        image_footnote_lines.sort(key=lambda x: np.min(np.array(x['bbox'])[:, 1]))
        for footnote_data in image_footnote_lines:
            cleaned_footnote_text = remove_keywords(footnote_data['text'], KEYWORDS_TO_REMOVE)
            if cleaned_footnote_text: # Only add if text is not empty after cleaning
                csv_data.append([
                    abbreviated_filename,
                    'FOOTNOTE', # Indicate this row is a footnote
                    '',         # Empty for Column
                    '',         # Empty for Text_Line
                    cleaned_footnote_text
                ])

        processed_images_count += 1

    except Exception as e:
        print(f"Error processing {filename}: {e}")
        import traceback
        traceback.print_exc() # Print full traceback for unhandled errors

print(f"\n--- Finished OCR for {processed_images_count} images. ---")

# --- Step 5: Write collected data to CSV ---
print(f"\n--- Writing results to CSV: {csv_file_path} ---")
with open(csv_file_path, 'w', newline='', encoding='utf-8') as csvfile:
    csv_writer = csv.writer(csvfile)
    csv_writer.writerows(csv_data)

print(f"\nOCR process complete! Results saved to: {csv_file_path}")

# Optional: Display the first few rows of the CSV to verify
print("\n--- First 5 rows of the generated CSV: ---")
with open(csv_file_path, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i < 5:
            print(line.strip())
        else:
            break

print("\n--- To download the CSV file, go to the 'MyDrive' tab on the left pane in Colab ---")
print(f"--- Then navigate to '/content/drive/MyDrive/连环画/Ref002/CSV' and download '{os.path.basename(output_csv_filename)}' ---")


--- Initializing PaddleOCR (this may download models if first run) ---


/usr/local/lib/python3.11/dist-packages/paddle/utils/cpp_extension/extension_utils.py:715: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-LCNet_x1_0_doc_ori', None)
Using official model (PP-LCNet_x1_0_doc_ori), the model files will be automatically downloaded and saved in `/root/.paddlex/official_models/PP-LCNet_x1_0_doc_ori`.
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional t

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

README.md: 0.00B [00:00, ?B/s]

inference.json: 0.00B [00:00, ?B/s]

...

inference.yml: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

inference.json: 0.00B [00:00, ?B/s]

inference.pdiparams:   0%|          | 0.00/84.4M [00:00<?, ?B/s]


--- Starting OCR for images in: /content/drive/MyDrive/连环画/Ref002/JPEG/Core ---
Processing: ref002_zglhhsc_0009_001.jpeg
Processing: ref002_zglhhsc_0010_002.jpeg
Processing: ref002_zglhhsc_0011_003.jpeg
Processing: ref002_zglhhsc_0012_004.jpeg
Processing: ref002_zglhhsc_0013_005.jpeg
Processing: ref002_zglhhsc_0014_006.jpeg
Processing: ref002_zglhhsc_0015_007.jpeg
  Identified ROI1 inner footer text (footnote): '3' (min_y=1352, ROI1_footer_threshold_y=1307)
Processing: ref002_zglhhsc_0016_008.jpeg
Processing: ref002_zglhhsc_0017_009.jpeg
Processing: ref002_zglhhsc_0018_010.jpeg
  Identified ROI1 inner footer text (footnote): '*该书仅出版上册。' (min_y=1334, ROI1_footer_threshold_y=1290)
Processing: ref002_zglhhsc_0019_011.jpeg
  Identified ROI1 inner footer text (footnote): '*该书版权页无定价。' (min_y=1360, ROI1_footer_threshold_y=1303)
Processing: ref002_zglhhsc_0020_012.jpeg
Processing: ref002_zglhhsc_0021_013.jpeg
Processing: ref002_zglhhsc_0022_014.jpeg
Processing: ref002_zglhhsc_0023_015.jpeg
Pr

### Raw OCR Output for `Copy of ref002_zglhhsc_0215_207.jpeg`

This cell will display all text lines detected by PaddleOCR for the specified image, along with their bounding box coordinates and confidence scores, before any custom filtering or grouping is applied. This helps to determine if PaddleOCR is failing to detect the text at its source.

In [ ]:
import os
from paddleocr import PaddleOCR
from PIL import Image
import numpy as np

# Re-initialize OCR if it's not in the kernel state (though it should be from previous runs)
# This is a safe guard if the kernel restarts or this cell is run independently.
if 'ocr' not in locals() or 'ocr' not in globals():
    print("Re-initializing PaddleOCR...")
    ocr = PaddleOCR(use_textline_orientation=True, lang="ch")

# Define the specific image path
input_image_folder = '/content/drive/MyDrive/连环画/Ref002/JPEG/Core'
specific_image_filename = 'Copy of ref002_zglhhsc_0215_207.jpeg'
image_path = os.path.join(input_image_folder, specific_image_filename)

print(f"\n--- Running raw OCR on: {specific_image_filename} ---")

try:
    # Get image dimensions (useful for visual context)
    with Image.open(image_path) as img:
        image_width, image_height = img.size
    print(f"Image dimensions: {image_width}x{image_height}")

    raw_ocr_output = ocr.predict(image_path)

    if raw_ocr_output and isinstance(raw_ocr_output, list) and len(raw_ocr_output) > 0:
        print("\nDetected text lines (raw output from PaddleOCR):")
        for page_result_dict in raw_ocr_output:
            if isinstance(page_result_dict, dict) and \
              'rec_polys' in page_result_dict and \
              'rec_texts' in page_result_dict and \
              'rec_scores' in page_result_dict:

                rec_polys = page_result_dict['rec_polys']
                rec_texts = page_result_dict['rec_texts']
                rec_scores = page_result_dict['rec_scores']

                for i in range(len(rec_polys)):
                    bounding_box = rec_polys[i]
                    text = rec_texts[i]
                    confidence = rec_scores[i]

                    points = np.array(bounding_box)
                    min_x, min_y = np.min(points, axis=0)
                    max_x, max_y = np.max(points, axis=0)

                    print(f"  Text: '{text}'")
                    print(f"    BBox: ([{min_x}, {min_y}], [{max_x}, {max_y}])")
                    print(f"    Confidence: {confidence:.4f}")
                    print(f"    Min_Y: {min_y}")
            else:
                print(f"  Warning: Unexpected dictionary structure in raw OCR output for {specific_image_filename}.")
    else:
        print(f"  No text detected by PaddleOCR for {specific_image_filename}.")

except Exception as e:
    print(f"Error processing {specific_image_filename}: {e}")
    import traceback
    traceback.print_exc()


--- Running raw OCR on: Copy of ref002_zglhhsc_0215_207.jpeg ---
Error processing Copy of ref002_zglhhsc_0215_207.jpeg: [Errno 2] No such file or directory: '/content/drive/MyDrive/连环画/Ref002/JPEG/Core/Copy of ref002_zglhhsc_0215_207.jpeg'


Traceback (most recent call last):
  File "/tmp/ipython-input-27-1578210789.py", line 21, in <cell line: 0>
    with Image.open(image_path) as img:
         ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/PIL/Image.py", line 3505, in open
    fp = builtins.open(filename, "rb")
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/连环画/Ref002/JPEG/Core/Copy of ref002_zglhhsc_0215_207.jpeg'


In [ ]:
import os
import csv
from paddleocr import PaddleOCR
from PIL import Image
import numpy as np # Used for convenient bounding box calculations
import requests
import re

# --- Configuration ---
input_image_folder = '/content/drive/MyDrive/连环画/Ref002/JPEG/Core'
output_csv_filename = '/content/drive/MyDrive/连环画/Ref002/CSV/OCR_Raw_Test'

# Define supported image extensions
IMAGE_EXTENSIONS = ('.png', '.jpg', '.jpeg', '.bmp', '.tiff', '.webp')

# Threshold for grouping lines vertically (in pixels)
# If the difference in top Y-coordinates of consecutive lines is within this threshold, they are grouped.
GROUPING_THRESHOLD_Y = 20

KEYWORDS_TO_REMOVE = [
]

def remove_keywords(text, keywords):
    cleaned_text = text
    for keyword in keywords:
        # Create a regex pattern to match the whole word (case-insensitive)
        # \b ensures whole word match, re.IGNORECASE for case-insensitivity
        pattern = r'\b' + re.escape(keyword) + r'\b'
        cleaned_text = re.sub(pattern, '', cleaned_text, flags=re.IGNORECASE).strip()
        # Remove any extra spaces left by removal (e.g., "word1  word2")
        cleaned_text = re.sub(r'\s+', ' ', cleaned_text).strip()
    return cleaned_text

# Helper function to get min/max coordinates from a bounding box
def get_bbox_coords(bbox_points):
    points = np.array(bbox_points)
    min_x, min_y = np.min(points, axis=0)
    max_x, max_y = np.max(points, axis=0)
    return min_x, min_y, max_x, max_y

# Helper function to check if a bounding box's center is within an ROI
def is_bbox_center_in_roi(bbox, roi):
    _, min_y, _, max_y = get_bbox_coords(bbox)
    bbox_center_y = (min_y + max_y) / 2
    # ROI is [roi_x_start, roi_y_start, roi_x_end, roi_y_end]
    return roi[1] <= bbox_center_y <= roi[3]

# --- Step 2: Initialize PaddleOCR ---
# 'lang="ch"' loads the Chinese model which handles both simplified and traditional.
# 'use_textline_orientation=True' improves accuracy by correcting text orientation.
print("\n--- Initializing PaddleOCR (this may download models if first run) ---")
ocr = PaddleOCR(use_textline_orientation=True, lang="ch")

# --- Step 3: Prepare CSV file for writing ---
# The output CSV file path should be joined to /content/drive/MyDrive, not just /content/
csv_file_path = os.path.join('/content/drive/MyDrive/连环画/Ref002/CSV', output_csv_filename.split('/')[-1])
csv_data = []
# Updated CSV header to include 'Column' and 'Footnote_Text' for multi-column layout and footnotes
csv_header = ['Page_Number', 'ROI', 'Column', 'Text_Line']
csv_data.append(csv_header)

# --- Step 4: Iterate through images, perform OCR, and collect data ---
print(f"\n--- Starting OCR for images in: {input_image_folder} ---")
processed_images_count = 0


# Get list of all files in the directory
all_files_in_folder = os.listdir(input_image_folder)

# Filter and SORT the image files
image_files = sorted([f for f in all_files_in_folder if f.lower().endswith(IMAGE_EXTENSIONS)])

for filename in image_files: # Use the sorted list here
    image_path = os.path.join(input_image_folder, filename)
    print(f"Processing: {filename}")
    try:
        # Load image to get dimensions for ROI calculation
        with Image.open(image_path) as img:
            image_width, image_height = img.size

        # Define ROIs dynamically
        # ROI 1: Bottom 42% of the upper half
        # Upper half ends at image_height / 2
        # Bottom 42% means from image_height/2 * (1 - 0.42) to image_height/2
        roi1_y_start = image_height / 2 * (1 - 0.42)
        roi1_y_end = image_height / 2
        roi1 = [0, roi1_y_start, image_width, roi1_y_end] # [x_start, y_start, x_end, y_end]

        # ROI 2: Bottom 42% of the lower half
        # Lower half starts at image_height / 2 and ends at image_height
        # Bottom 42% means from image_height - (image_height/2 * 0.42) to image_height
        roi2_y_start = image_height - (image_height / 2 * 0.42)
        roi2_y_end = image_height
        roi2 = [0, roi2_y_start, image_width, roi2_y_end]

        ocr_output = ocr.predict(image_path)

        all_lines_raw = []

        if ocr_output and isinstance(ocr_output, list) and len(ocr_output) > 0:
            for page_result_dict in ocr_output:
                if isinstance(page_result_dict, dict) and \
                  'rec_polys' in page_result_dict and \
                  'rec_texts' in page_result_dict and \
                  'rec_scores' in page_result_dict:

                    rec_polys = page_result_dict['rec_polys']
                    rec_texts = page_result_dict['rec_texts']
                    rec_scores = page_result_dict['rec_scores']

                    for i in range(len(rec_polys)):
                        bounding_box = rec_polys[i]
                        text = rec_texts[i]
                        confidence = rec_scores[i]

                        # Calculate min_y for the bounding box
                        min_x, min_y, max_x, max_y = get_bbox_coords(bounding_box)

                        if not isinstance(text, str):
                            text = str(text)

                        all_lines_raw.append({
                            'bbox': bounding_box,
                            'text': text,
                            'confidence': confidence
                        })
                else:
                    print(f"  Warning: Unexpected dictionary structure in OCR output for {filename}. Skipping block.")
                    print(f"  Block content: {page_result_dict}")
        else:
            print(f"  Warning: OCR output is empty or not in expected list format for {filename}. No text extracted.")

        # Calculate column width for subdividing ROIs
        column_width = image_width / 4

        # Initialize lists for lines in each column within each ROI
        roi1_cols_lines = [[] for _ in range(4)] # For ROI1: Col1, Col2, Col3, Col4
        roi2_cols_lines = [[] for _ in range(4)] # For ROI2: Col1, Col2, Col3, Col4

        # Distribute lines into ROIs and then into columns
        for line_data in all_lines_raw:
            min_x, min_y, max_x, max_y = get_bbox_coords(line_data['bbox']) # Get all coords for min_y, max_y
            bbox_center_x = (min_x + max_x) / 2 # Use center X-coordinate for column assignment

            # Determine raw column index (0-3)
            col_idx = int(bbox_center_x / column_width)

            # NEW FILTERING: Skip line if its horizontal center is outside the 4 columns
            if not (0 <= col_idx < 4):
                print(f"  Skipping text horizontally outside columns: '{line_data['text']}' (bbox_center_x={bbox_center_x:.0f}, col_idx={col_idx})")
                continue # Skip this line as it's horizontally outside the expected columns

            # Check for ROI1
            if is_bbox_center_in_roi(line_data['bbox'], roi1):
                roi1_cols_lines[col_idx].append(line_data)
            # Check for ROI2
            elif is_bbox_center_in_roi(line_data['bbox'], roi2):
                roi2_cols_lines[col_idx].append(line_data)

        # --- Group lines within each column independently ---
        def group_lines_vertically(lines, threshold_y):
            if not lines:
                return []

            # Sort lines by their top-most y-coordinate first
            lines.sort(key=lambda x: np.min(np.array(x['bbox'])[:, 1]))

            grouped_lines = []
            if not lines: # Handle empty lines list after sorting
                return []

            current_group = [lines[0]]

            for i in range(1, len(lines)):
                prev_line_bbox = np.array(current_group[-1]['bbox'])
                current_line_bbox = np.array(lines[i]['bbox'])

                prev_min_y = np.min(prev_line_bbox[:, 1])
                current_min_y = np.min(current_line_bbox[:, 1])

                if abs(current_min_y - prev_min_y) <= threshold_y:
                    current_group.append(lines[i])
                else:
                    grouped_lines.append(current_group)
                    current_group = [lines[i]]

            if current_group: # Add the last group
                grouped_lines.append(current_group)

            merged_output = []
            for group in grouped_lines:
                # Sort lines within the current group by their X-coordinate
                group.sort(key=lambda x: np.min(np.array(x['bbox'])[:, 0]))

                merged_text = " ".join([line['text'] for line in group]) # Join with space
                merged_confidence = np.mean([line['confidence'] for line in group]) # Average confidence

                # Calculate the merged bounding box
                all_x_coords = []
                all_y_coords = []
                for line in group:
                    bbox_points = np.array(line['bbox'])
                    all_x_coords.extend(bbox_points[:, 0])
                    all_y_coords.extend(bbox_points[:, 1])

                min_x, min_y = np.min(all_x_coords), np.min(all_y_coords)
                max_x, max_y = np.max(all_x_coords), np.max(all_y_coords)

                merged_output.append({
                    'bbox': [[min_x, min_y], [max_x, min_y], [max_x, max_y], [min_x, max_y]],
                    'text': merged_text,
                    'confidence': merged_confidence
                })
            return merged_output

        # List to store all final processed lines for the current image
        final_processed_lines_for_csv = []

        # Process ROI1 columns
        for i, col_lines in enumerate(roi1_cols_lines):
            grouped_col_lines = group_lines_vertically(col_lines, GROUPING_THRESHOLD_Y)
            for line_info in grouped_col_lines:
                line_info['roi_label'] = 'ROI1'
                line_info['column_label'] = f'Col{i+1}'
                final_processed_lines_for_csv.append(line_info)

        # Process ROI2 columns
        for i, col_lines in enumerate(roi2_cols_lines):
            grouped_col_lines = group_lines_vertically(col_lines, GROUPING_THRESHOLD_Y)
            for line_info in grouped_col_lines:
                line_info['roi_label'] = 'ROI2'
                line_info['column_label'] = f'Col{i+1}'
                final_processed_lines_for_csv.append(line_info)

        # Prepare abbreviated filename
        last_dot_index = filename.rfind('.')
        filename_without_extension = filename[:last_dot_index] if last_dot_index != -1 else filename
        abbreviated_filename = filename_without_extension[-3:]

        # Add results to csv_data from the fully processed lines (main content)
        for line_data in final_processed_lines_for_csv:
            cleaned_text = remove_keywords(line_data['text'], KEYWORDS_TO_REMOVE)
            if cleaned_text: # Only add the line if the text is not empty after cleaning
                csv_data.append([
                    abbreviated_filename,
                    line_data['roi_label'],
                    line_data['column_label'],
                    cleaned_text
                ])

        processed_images_count += 1

    except Exception as e:
        print(f"Error processing {filename}: {e}")
        import traceback
        traceback.print_exc() # Print full traceback for unhandled errors

print(f"\n--- Finished OCR for {processed_images_count} images. ---")

# --- Step 5: Write collected data to CSV ---
print(f"\n--- Writing results to CSV: {csv_file_path} ---")
with open(csv_file_path, 'w', newline='', encoding='utf-8') as csvfile:
    csv_writer = csv.writer(csvfile)
    csv_writer.writerows(csv_data)

print(f"\nOCR process complete! Results saved to: {csv_file_path}")

# Optional: Display the first few rows of the CSV to verify
print("\n--- First 5 rows of the generated CSV: ---")
with open(csv_file_path, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i < 5:
            print(line.strip())
        else:
            break

print("\n--- To download the CSV file, go to the 'MyDrive' tab on the left pane in Colab ---")
print(f"--- Then navigate to '/content/drive/MyDrive/连环画/Ref002/CSV' and download '{os.path.basename(output_csv_filename)}' ---")